RECON assignment V2

In [ ]:
import numpy as np
import pydicom
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1) DICOM beolvasás → 2D sinogram (projection × detector)
# ------------------------------------------------------------
def load_projections(dicom_path):
    """
    DICOM fájl beolvasása.
    A Mediso SPECT DICOM 3D tömböt tartalmaz:
        (frames, rows, cols) = (vetületek, 256, 256)

    A rekonstrukcióhoz viszont 1D vonal szükséges minden vetületből,
    ezért minden frame-ből a KÖZÉPSŐ sort vesszük ki → (num_angles, num_detectors)
    """
    ds = pydicom.dcmread(dicom_path)

    # Pixeladatok kinyerése float32 formában
    arr = ds.pixel_array.astype(np.float32)
    print(f"{dicom_path} raw shape:", arr.shape)

    # Biztosítsuk, hogy 3 dimenziós
    if arr.ndim != 3:
        raise ValueError(f"Várt 3D tömb, de ez érkezett: {arr.shape}")

    num_frames, rows, cols = arr.shape

    # A detektor középső sora minden vetületben
    mid_row = rows // 2
    projections = arr[:, mid_row, :]

    print("használt projections shape:", projections.shape)  # (120, 256)
    return projections, ds


# ------------------------------------------------------------
# 2) Ramp filter (klasszikus FBP előszűrés)
#    + opcionális Hann-ablak apodizálás
# ------------------------------------------------------------
def ramp_filter(projections, use_hann=False):
    """
    Filtered Backprojection szűrőmagja.
    - Ramp filter: |ω|
    - Hann window (opcionális): csökkenti a zajt és a csillagszerű artefaktumokat.
    """
    num_angles, num_detectors = projections.shape

    # Frekvenciák a Nyquist tartomány feléig
    freqs = np.fft.rfftfreq(num_detectors).reshape(1, -1)

    # Ramp filter alapja
    ramp = np.abs(freqs)

    # Ha kérjük: Hann ablak alkalmazása a ramp filterre
    if use_hann:
        # Normalizálás 0..1 közé
        f_norm = freqs / freqs.max()
        hann = 0.5 * (1 - np.cos(2 * np.pi * f_norm))
        filter_kernel = ramp * hann
    else:
        filter_kernel = ramp

    # FFT vetületenként
    proj_fft = np.fft.rfft(projections, axis=1)

    # Szűrés frekvenciatartományban
    proj_fft_filtered = proj_fft * filter_kernel

    # Inverz FFT → vissza detektor síkra
    filtered = np.fft.irfft(proj_fft_filtered, n=num_detectors, axis=1)
    return filtered


# ------------------------------------------------------------
# 3) Egyetlen vetület visszavetítése (paralell fut)
# ------------------------------------------------------------
def backproject_single(args):
    """
    Egy vetület (egy szög) hozzájárulásának kiszámítása.
    A FBP során minden vetület "szétkenődik" a képen.
    Ezt a műveletet párhuzamosan futtatjuk több processben.
    """
    idx, theta, projection, x_grid, y_grid, det_positions = args

    # Koordináta-transzformáció:
    # egy (x, y) pixel melyik detektor t pozícióra vetül?
    t = x_grid * np.cos(theta) + y_grid * np.sin(theta)

    # Interpoláció: detektor adat → kép rács
    contrib = np.interp(
        t.ravel(),
        det_positions,
        projection,
        left=0.0,
        right=0.0
    ).reshape(x_grid.shape)

    return contrib


# ------------------------------------------------------------
# 4) FBP rekonstrukció párhuzamosítva
# ------------------------------------------------------------
def reconstruct_fbp_parallel(projections, angles_rad, img_size=256, use_hann=False):
    """
    Teljes FBP rekonstrukció:
    - koordináta-rács létrehozása,
    - ramp (Hann) szűrés,
    - visszavetítés minden szögre (ProcessPoolExecutor),
    - átlagolás.
    """
    num_angles, num_detectors = projections.shape

    # Rekonstrukciós rács (x,y): [-1, 1] tartomány
    x = np.linspace(-1.0, 1.0, img_size)
    y = np.linspace(-1.0, 1.0, img_size)
    x_grid, y_grid = np.meshgrid(x, y)

    # Detektorsík pozíciói (szintén -1..1-be skálázva)
    det_positions = np.linspace(-1.0, 1.0, num_detectors)

    # Előszűrés ramp + opcionális Hann
    filtered_projections = ramp_filter(projections, use_hann=use_hann)

    # Munkacsomag előkészítése minden szögre
    task_args = [
        (i, angles_rad[i], filtered_projections[i], x_grid, y_grid, det_positions)
        for i in range(num_angles)
    ]

    # Kimeneti kép kezdésnek 0
    recon = np.zeros_like(x_grid, dtype=np.float32)

    # Párhuzamos visszavetítés
    with ProcessPoolExecutor() as ex:
        for contrib in ex.map(backproject_single, task_args):
            recon += contrib

    # Átlagolás vetületszámmal
    recon /= num_angles
    return recon


# ------------------------------------------------------------
# 5) Szögek kinyerése DICOM headerből
# ------------------------------------------------------------
def get_angles_from_dicom(ds, num_angles):
    """
    A forgási szögek kinyerése a DICOM metaadatból.
    Ha nem találjuk, fallback: 0..π között egyenletes eloszlás.
    """
    import numpy as np
    try:
        # Forgási információ (Rotation Information Sequence)
        rot_seq = ds[0x0054, 0x0052][0]

        start_angle_deg = float(rot_seq[0x0054, 0x0200].value)   # kezdőszög
        step_deg = float(rot_seq[0x0018, 0x1144].value)          # lépésköz

        # Szögvektor generálása
        angles_deg = start_angle_deg + np.arange(num_angles) * step_deg
        angles_rad = np.deg2rad(angles_deg)

        # FBP-ben periodicitás miatt elég 0..π tartomány
        angles_rad = np.mod(angles_rad, np.pi)

        print("Szögek beolvasva DICOM-ból. Első 5:", angles_rad[:5])
    except Exception as e:
        print("Nem sikerült szögeket kinyerni, fallback 0..π", e)
        angles_rad = np.linspace(0, np.pi, num_angles, endpoint=False)

    return angles_rad


# ------------------------------------------------------------
# 6) Főprogram
# ------------------------------------------------------------
def main():
    # Két különböző zajszintű DICOM beolvasása
    proj1, ds1 = load_projections("jaszczak_lehr-hs_130mm_main_VBProjection_signals.dcm")
    proj2, ds2 = load_projections("jaszczak_lehr-hs_130mm_main_VBProjection_signals_seed31337_120m.dcm")

    # Szögek előállítása a DICOM metaadatból
    num_angles = proj1.shape[0]
    angles_rad = get_angles_from_dicom(ds1, num_angles)

    # 1) Eredeti (nyers ramp filter)
    recon1_plain = reconstruct_fbp_parallel(proj1, angles_rad, img_size=256, use_hann=False)
    recon2_plain = reconstruct_fbp_parallel(proj2, angles_rad, img_size=256, use_hann=False)

    # 2) Javított (Hann-ablakos ramp)
    recon1_hann = reconstruct_fbp_parallel(proj1, angles_rad, img_size=256, use_hann=True)
    recon2_hann = reconstruct_fbp_parallel(proj2, angles_rad, img_size=256, use_hann=True)

    # Megjelenítés: felül az eredeti, alul a javított
    fig, axs = plt.subplots(2, 2, figsize=(8, 8))

    axs[0, 0].imshow(recon1_plain, cmap="gray")
    axs[0, 0].set_title("Eredeti FBP - alap jel")
    axs[0, 0].axis("off")

    axs[0, 1].imshow(recon2_plain, cmap="gray")
    axs[0, 1].set_title("Eredeti FBP - seed31337")
    axs[0, 1].axis("off")

    axs[1, 0].imshow(recon1_hann, cmap="gray")
    axs[1, 0].set_title("Hann FBP - alap jel")
    axs[1, 0].axis("off")

    axs[1, 1].imshow(recon2_hann, cmap="gray")
    axs[1, 1].set_title("Hann FBP - seed31337")
    axs[1, 1].axis("off")

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Program indítása
# ------------------------------------------------------------
if __name__ == "__main__":
    main()


In [ ]:
if __name__ == "__main__":
    main()

![v1](pic_v2.png)